# SumExplicitStack (Python)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_python.ipynb)

Capitolo 18 — Algoritmi ricorsivi e iterativi a confronto. Notebook eseguibile di `SumExplicitStack`: l'algoritmo in Python, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Lo stesso notebook in [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_cpp.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo/SumExplicitStack_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_python.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Python 3, senza installare nulla.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fib_rec` (snake_case).

## Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [1]:
max_p = 0   # massima dimensione della pila P

def sum_explicit_stack(a, n):
    """SumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita."""
    global max_p
    s = 0
    P = []                          # pila dei sotto-array da sommare
    P.append((0, n - 1))
    max_p = max(max_p, len(P))
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]            # caso base
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            max_p = max(max_p, len(P))
    return s

def fmt_p(P):
    return "[" + ",".join(f"({x},{y})" for x, y in P) + "]"

def sum_explicit_stack_trace(a, n):
    s = 0
    P = [(0, n - 1)]
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]
            print(f"pop ({l},{r}): caso base, s={s} => P={fmt_p(P)}")
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            print(f"pop ({l},{r}): mid={mid}, push ({mid + 1},{r}) e ({l},{mid}) => P={fmt_p(P)}, s={s}")
    return s

# ---- le altre somme del codice del corso, con la profondità ----
depth = 0
max_depth = 0

def sum_it(a, n):
    s = 0
    for i in range(n):
        s = s + a[i]
    return s

def sum_rec_halving(a, l, r):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    depth -= 1
    return result

def sum_rec_decrease(a, n):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if n == 0:
        result = 0
    else:
        result = sum_rec_decrease(a, n - 1) + a[n - 1]
    depth -= 1
    return result

a = [4, 7, 1, 9]
n = len(a)
print("a =", a, "  n =", n)
print("SumExplicitStack(a, 4):")
res = sum_explicit_stack_trace(a, n)
max_p = 0
sum_explicit_stack(a, n)
print(f"risultato {res}, dimensione massima di P = {max_p}")
print()
print("sumIt                =", sum_it(a, n))
max_depth = 0
print("sumRecHalving        =", sum_rec_halving(a, 0, n - 1), "  profondità massima =", max_depth)
max_depth = 0
print("sumRecDecrease       =", sum_rec_decrease(a, n), "  profondità massima =", max_depth)

# ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
step = 0
max_active = 0

def emit(stack, action):
    global step, max_active
    step += 1
    max_active = max(max_active, len(stack))
    print(f"{step:>2d} | {' '.join(stack)} | {action}")

def sum_rec_decrease_trace(a, n, stack):
    stack.append(f"s({n})")
    if n == 0:
        emit(stack, "n=0: return 0")
        stack.pop()
        return 0
    emit(stack, f"n={n} != 0: chiama s({n - 1}), poi somma a[{n - 1}]")
    r = sum_rec_decrease_trace(a, n - 1, stack)
    emit(stack, f"return {r}+a[{n - 1}]={r + a[n - 1]}")
    stack.pop()
    return r + a[n - 1]

b = [4, 7, 1]
print()
print("a =", b, "  sumRecDecrease(a, 3)")
print(" # | pila delle chiamate attive | azione")
res = sum_rec_decrease_trace(b, 3, [])
print(f"risultato {res}, chiamate attive al massimo = {max_active} = n+1")

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [2]:
# Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
c = list(range(1, 101))
max_p = 0
print(sum_explicit_stack(c, len(c)), max_p)

5050 8


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).